In [2]:
import io
import time
import pandas as pd
import urllib3
import requests

In [ ]:
# suppress SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning) #nvm

In [ ]:
import os
path = "../certs/emsign_root_ca_g1.pem" # permanent fix to SSL error

if os.path.exists(path):
    print("exists")
else:
    print("nope")

exists


In [ ]:
# Test subset: 1 (Dry Maize), 3 (Wheat)
TEST_COMMODITIES = {
    1: "Dry Maize",
    3: "Wheat",
}

BASE_URL = "https://kamis.kilimo.go.ke/site/market"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko)"
        " Chrome/118.0.0.0 Safari/537.36"
    )
}

frames = []

for pid, name in TEST_COMMODITIES.items():
    print(f"Fetching {name} (ID: {pid})...")
    params = {
        "product": pid,
        "per_page": 20,  
        "export": "excel",
    }

    try:
        response = requests.get(
            BASE_URL, params=params, headers=HEADERS, verify="../certs/emsign_root_ca_g1.pem", timeout=15 
        )
        response.raise_for_status()

        # Some web servers return HTML error pages disguised with 200 OK
        if (
            b"<html" in response.content[:100].lower()
            or b"<!doctype" in response.content[:100].lower()
        ):
            print(
                f"Warning: Response for {name} appears to be HTML rather than Excel binary."
            )
            # Fallback to HTML table parser if they served an HTML table instead of xlsx
            df_list = pd.read_html(response.text)
            df = df_list[0] if df_list else pd.DataFrame()
        else:
            # Parse the Excel binary directly from memory
            df = pd.read_excel(io.BytesIO(response.content))

        if not df.empty:
            df["commodity_id"] = pid
            df["commodity_name"] = name
            frames.append(df)
            print(f"-> Success: Retrieved {len(df)} rows.")
        else:
            print(f"-> Empty dataset returned for {name}.")

    except Exception as err:
        print(f"Failed to fetch {name}: {err}")

    time.sleep(1)

# Inspect the merged output
if frames:
    result_df = pd.concat(frames, ignore_index=True)
    display(result_df.head(10))
    print("\nData Types & Columns:")
    result_df.info()
else:
    print("No data retrieved.")

Fetching Dry Maize (ID: 1)...
-> Success: Retrieved 20 rows.
Fetching Wheat (ID: 3)...
-> Success: Retrieved 20 rows.


,Commodity,Classification,Grade,Sex,Market,Wholesale,Retail,Supply Volume,County,Date,commodity_id,commodity_name
0,Dry Maize,White Maize,-,-,Kawangware,51.11/Kg,60.00/Kg,NaN,Nairobi,2026-10-19,1,Dry Maize
1,Dry Maize,White Maize,-,-,Ngurubani Market,46.67/Kg,50.00/Kg,4520.0,Kirinyaga,2026-10-19,1,Dry Maize
2,Dry Maize,White Maize,-,-,Butere Livestock Market,44.44/Kg,52.17/Kg,9000.0,Kakamega,2026-10-19,1,Dry Maize
3,Dry Maize,White Maize,-,-,Kalundu,42.22/Kg,50.00/Kg,NaN,Kitui,2026-10-19,1,Dry Maize
4,Dry Maize,White Maize,-,-,Eldoret Main,100.00/Kg,110.00/Kg,270.0,Uasin-Gishu,2026-10-19,1,Dry Maize
5,Dry Maize,White Maize,-,-,Isebania Market,-,48.00/Kg,10000.0,Migori,2026-10-18,1,Dry Maize
6,Dry Maize,Yellow Maize,-,-,Kibuye,-,65.00/Kg,NaN,Kisumu,2026-10-18,1,Dry Maize
7,Dry Maize,White Maize,-,-,Kibuye,53.33/Kg,65.00/Kg,NaN,Kisumu,2026-10-18,1,Dry Maize
8,Dry Maize,White Maize,-,-,Khayega,44.44/Kg,48.89/Kg,10800.0,Kakamega,2026-10-18,1,Dry Maize
9,Dry Maize,Mixed-Traditional,-,-,Taveta Retail Market,35.00/Kg,50.00/Kg,9000.0,Taita-Taveta,2026-10-17,1,Dry Maize



Data Types & Columns:
<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Commodity       40 non-null     str    
 1   Classification  40 non-null     str    
 2   Grade           40 non-null     str    
 3   Sex             40 non-null     str    
 4   Market          40 non-null     str    
 5   Wholesale       40 non-null     str    
 6   Retail          40 non-null     str    
 7   Supply Volume   29 non-null     float64
 8   County          40 non-null     str    
 9   Date            40 non-null     str    
 10  commodity_id    40 non-null     int64  
 11  commodity_name  40 non-null     str    
dtypes: float64(1), int64(1), str(10)
memory usage: 3.9 KB


In [13]:
result_df['commodity_name'].nunique()

2

In [15]:
result_df['commodity_name'].unique()

<StringArray>
['Dry Maize', 'Wheat']
Length: 2, dtype: str

In [14]:
result_df.tail()

,Commodity,Classification,Grade,Sex,Market,Wholesale,Retail,Supply Volume,County,Date,commodity_id,commodity_name
35,Wheat,-,-,-,Eldoret Main,90.00/Kg,100.00/Kg,270.0,Uasin-Gishu,2026-10-06,3,Wheat
36,Wheat,-,-,-,Kawangware,77.78/Kg,90.00/Kg,NaN,Nairobi,2026-10-05,3,Wheat
37,Wheat,-,-,-,Kibuye,56.67/Kg,65.00/Kg,NaN,Kisumu,2026-10-04,3,Wheat
38,Wheat,-,-,-,Cheptiret - Uasin Gishu,50.00/Kg,85.00/Kg,1000.0,Uasin-Gishu,2026-10-02,3,Wheat
39,Wheat,-,-,-,Kawangware,72.22/Kg,90.00/Kg,NaN,Nairobi,2026-10-02,3,Wheat


In [2]:
URL = ("https://opendata.knbs.or.ke/beta/databrowserhub/api/Core/nodes/1/"
       "datasets/KE1,DF_CPI_IX,1.0/data")
HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) Chrome/154.0.0.0",
           "Accept": "application/json", "Referer": "https://opendata.knbs.or.ke/beta/databrowser/"}

def fetch_cpi(payload=None):
    r = (requests.post(URL, json=payload, headers=HEADERS, timeout=60) if payload
         else requests.get(URL, headers=HEADERS, timeout=60))
    r.raise_for_status()
    return r.json()


In [ ]:
import json, requests, pandas as pd
from pathlib import Path
from datetime import datetime, timezone

URL = ("https://opendata.knbs.or.ke/beta/databrowserhub/api/Core/nodes/1/"
       "datasets/KE1,DF_CPI_IX,1.0/data")
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) Chrome/154.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://opendata.knbs.or.ke/beta/databrowser/",
}
PAYLOAD = [{"id": "FREQ", "filterValues": ["M"], "type": "CodeValues", "period": 0}]

def fetch_cpi_raw():
    r = requests.post(URL, json=PAYLOAD, headers=HEADERS, timeout=60)
    r.raise_for_status()
    return r.json()                      

def save_bronze(js, root="bronze/knbs"):
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    p = Path(root) / f"cpi_{ts}.json"
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(js))
    return p

def jsonstat_to_df(js):
    ids, sizes = js["id"], js["size"]    # ['FREQ','ITEM_PRICE','UNIT_MEASURE','TIME_PERIOD'], [1,14,1,90]
    labels = {}
    for d in ids:
        cat = js["dimension"][d]["category"]
        labels[d] = [(c, cat["label"].get(c, c)) for c in cat["index"]]

    rows = []
    for k, v in js["value"].items():
        i, pos = int(k), []
        for s in reversed(sizes):
            pos.append(i % s); i //= s
        pos.reverse()
        rec = {}
        for d, p in zip(ids, pos):
            code, label = labels[d][p]
            rec[d.lower()] = code
            rec[d.lower() + "_label"] = label
        rec["cpi_index"] = v
        rows.append(rec)
    df = pd.DataFrame(rows)
    df["period_start"] = pd.to_datetime(df["time_period"] + "-01")
    return df


In [ ]:
js = fetch_cpi_raw()
save_bronze(js)

In [ ]:
df = jsonstat_to_df(js)
print(df.shape)

(1260, 10)


In [7]:
df[df.item_price == "CP01"].tail(3)

,freq,freq_label,item_price,item_price_label,unit_measure,unit_measure_label,time_period,time_period_label,cpi_index,period_start
177,M,Monthly,CP01,FOOD AND NON-ALCOHOLIC BEVERAGES,IX,Index,2026-05,2026-05,193.900000,2026-05-01
178,M,Monthly,CP01,FOOD AND NON-ALCOHOLIC BEVERAGES,IX,Index,2026-06,2026-06,194.400000,2026-06-01
179,M,Monthly,CP01,FOOD AND NON-ALCOHOLIC BEVERAGES,IX,Index,2026-07,2026-07,194.612627,2026-07-01


In [3]:
path = "../data/bronze/kamis/2026-10-20T101324Z/1/page_2.xlsx"
kamis_2 = pd.read_excel(path)

kamis_2.head()

,Commodity,Classification,Grade,Sex,Market,Wholesale,Retail,Supply Volume,County,Date
0,Dry Maize,White Maize,-,-,Chebunyo,53.33/Kg,65.00/Kg,18000.0,Bomet,2026-10-19
1,Dry Maize,White Maize,-,-,Kawangware,51.11/Kg,60.00/Kg,NaN,Nairobi,2026-10-19
2,Dry Maize,White Maize,-,-,Ngurubani Market,46.67/Kg,50.00/Kg,4520.0,Kirinyaga,2026-10-19
3,Dry Maize,White Maize,-,-,Butere Livestock Market,44.44/Kg,52.17/Kg,9000.0,Kakamega,2026-10-19
4,Dry Maize,White Maize,-,-,Kalundu,42.22/Kg,50.00/Kg,NaN,Kitui,2026-10-19


In [4]:
kamis_2['Date'].nunique()

210

In [5]:
path = "../data/bronze/kamis/2026-10-20T101324Z/1/page_1.xlsx"
kamis_1 = pd.read_excel(path)

kamis_1.head()

,Commodity,Classification,Grade,Sex,Market,Wholesale,Retail,Supply Volume,County,Date
0,Dry Maize,White Maize,-,-,Chebunyo,53.33/Kg,65.00/Kg,18000.0,Bomet,2026-10-19
1,Dry Maize,White Maize,-,-,Kawangware,51.11/Kg,60.00/Kg,NaN,Nairobi,2026-10-19
2,Dry Maize,White Maize,-,-,Ngurubani Market,46.67/Kg,50.00/Kg,4520.0,Kirinyaga,2026-10-19
3,Dry Maize,White Maize,-,-,Butere Livestock Market,44.44/Kg,52.17/Kg,9000.0,Kakamega,2026-10-19
4,Dry Maize,White Maize,-,-,Kalundu,42.22/Kg,50.00/Kg,NaN,Kitui,2026-10-19


In [6]:
kamis_1['Date'].nunique()

210

In [8]:
path = "Market Prices Search.xls"
kamis_new = pd.read_excel(path)

kamis_new.head()

,Market,Commodity,Classification,Grade,Sex,Wholesale,Retail,Supply Volume,County,Date
0,Cheptiret - Uasin Gishu,Dry Maize,White Maize,-,-,55.56/Kg,60.00/Kg,5000,Uasin-Gishu,2026-10-14
1,Cheptiret - Uasin Gishu,Dry Maize,White Maize,-,-,45.45/Kg,60.00/Kg,5000,Uasin-Gishu,2026-10-09
2,Cheptiret - Uasin Gishu,Dry Maize,White Maize,-,-,55.56/Kg,60.00/Kg,5000,Uasin-Gishu,2026-10-07
3,Cheptiret - Uasin Gishu,Dry Maize,White Maize,-,-,45.45/Kg,60.00/Kg,5000,Uasin-Gishu,2026-10-02


In [7]:
# Check for verification behaviour
# copied from previous test , may behave differently due to new endpoint
# Test subset: 1 (Dry Maize), 3 (Wheat)
TEST_COMMODITIES = {
    1: "Dry Maize",
    3: "Wheat",
}

BASE_URL = "https://kamis.kilimo.go.ke/site/market_search/market_search"
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko)"
        " Chrome/118.0.0.0 Safari/537.36"
    )
}

frames = []

for pid, name in TEST_COMMODITIES.items():
    print(f"Fetching {name} (ID: {pid})...")
    params = {
        "product": pid,
        "per_page": 20,  
        "export": "excel",
    }

    try:
        response = requests.get(
            BASE_URL, params=params, headers=HEADERS,verify="../certs/emsign_root_ca_g1.pem", timeout=15
        )
        response.raise_for_status()

        # Some web servers return HTML error pages disguised with 200 OK
        if (
            b"<html" in response.content[:100].lower()
            or b"<!doctype" in response.content[:100].lower()
        ):
            print(
                f"Warning: Response for {name} appears to be HTML rather than Excel binary."
            )
            # Fallback to HTML table parser if they served an HTML table instead of xlsx
            df_list = pd.read_html(response.text)
            df = df_list[0] if df_list else pd.DataFrame()
        else:
            # Parse the Excel binary directly from memory
            df = pd.read_excel(io.BytesIO(response.content))

        if not df.empty:
            df["commodity_id"] = pid
            df["commodity_name"] = name
            frames.append(df)
            print(f"-> Success: Retrieved {len(df)} rows.")
        else:
            print(f"-> Empty dataset returned for {name}.")

    except Exception as err:
        print(f"Failed to fetch {name}: {err}")

    time.sleep(1)

# Inspect the merged output
if frames:
    result_df = pd.concat(frames, ignore_index=True)
    display(result_df.head(10))
    print("\nData Types & Columns:")
    result_df.info()
else:
    print("No data retrieved.") 

Fetching Dry Maize (ID: 1)...
-> Success: Retrieved 20 rows.
Fetching Wheat (ID: 3)...
-> Success: Retrieved 20 rows.


,Market,Commodity,Classification,Grade,Sex,Wholesale,Retail,Supply Volume,County,Date,commodity_id,commodity_name
0,Kagio,Lettuce,-,-,-,120.00/Kg,150.00/Kg,100.0,Kirinyaga,2026-10-22,1,Dry Maize
1,Kagio,Chillies,-,-,-,80.00/Kg,120.00/Kg,500.0,Kirinyaga,2026-10-22,1,Dry Maize
2,Kagio,Egg plant (Brinjals),-,-,-,20.00/Kg,30.00/Kg,150.0,Kirinyaga,2026-10-22,1,Dry Maize
3,Kagio,Spring Onions,-,-,-,30.00/Kg,50.00/Kg,350.0,Kirinyaga,2026-10-22,1,Dry Maize
4,Maua,Cassava Fresh,-,-,-,17.14/Kg,40.00/Kg,490.0,Meru,2026-10-22,1,Dry Maize
5,Maua,Yam,-,-,-,120.00/Kg,160.00/Kg,1400.0,Meru,2026-10-22,1,Dry Maize
6,Kawangware,Red Sorghum,-,-,-,68.89/Kg,75.00/Kg,NaN,Nairobi,2026-10-22,1,Dry Maize
7,Kawangware,Rice,Sindano,-,-,156.00/Kg,150.00/Kg,NaN,Nairobi,2026-10-22,1,Dry Maize
8,Maua,Sweet potatoes,-,-,-,45.00/Kg,80.00/Kg,600.0,Meru,2026-10-22,1,Dry Maize
9,Kawangware,Rice,Pishori,-,-,158.33/Kg,180.00/Kg,NaN,Nairobi,2026-10-22,1,Dry Maize



Data Types & Columns:
<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Market          40 non-null     str    
 1   Commodity       40 non-null     str    
 2   Classification  40 non-null     str    
 3   Grade           40 non-null     str    
 4   Sex             40 non-null     str    
 5   Wholesale       40 non-null     str    
 6   Retail          40 non-null     str    
 7   Supply Volume   24 non-null     float64
 8   County          40 non-null     str    
 9   Date            40 non-null     str    
 10  commodity_id    40 non-null     int64  
 11  commodity_name  40 non-null     str    
dtypes: float64(1), int64(1), str(10)
memory usage: 3.9 KB


In [9]:
from pathlib import Path
import pandas as pd

folder = Path("data/bronze/kamis_test")

for path in folder.rglob("*.xlsx"):
    if "diagnostics" in path.parts:
        df = pd.read_excel(path)

        repeated = df[
            (df["Market"] == "Kakamega Town")
            & (pd.to_datetime(df["Date"]).dt.date.astype(str) == "2026-10-07")
        ]

        if not repeated.empty:
            print(f"\nFile: {path}")
            print("Matching rows:", len(repeated))
            print(
                repeated[
                    ["Market", "Date", "Wholesale", "Retail", "Supply Volume"]
                ].to_string(index=False)
            )

In [12]:
from pathlib import Path
import pandas as pd

bronze_dir = Path("../data/bronze/kamis_test")

excel_files = list(bronze_dir.rglob("*.xlsx"))

print(f"Excel files found: {len(excel_files)}")

for file in excel_files:
    print(file)

Excel files found: 9
../data/bronze/kamis_test/2026-10-22T111939756978Z/1/accepted/2026-10-01_2026-10-08.xlsx
../data/bronze/kamis_test/2026-10-22T111939756978Z/1/accepted/2026-10-09_2026-10-15.xlsx
../data/bronze/kamis_test/2026-10-22T111939756978Z/1/diagnostics/2026-10-01_2026-10-15_capped.xlsx
../data/bronze/kamis_test/2026-10-22T112905699252Z/1/accepted/2026-10-01_2026-10-08.xlsx
../data/bronze/kamis_test/2026-10-22T112905699252Z/1/accepted/2026-10-09_2026-10-15.xlsx
../data/bronze/kamis_test/2026-10-22T112905699252Z/1/diagnostics/2026-10-01_2026-10-15_capped.xlsx
../data/bronze/kamis_test/2026-10-20T190301194678Z/1/accepted/2026-10-01_2026-10-08.xlsx
../data/bronze/kamis_test/2026-10-20T190301194678Z/1/accepted/2026-10-09_2026-10-15.xlsx
../data/bronze/kamis_test/2026-10-20T190301194678Z/1/diagnostics/2026-10-01_2026-10-15_capped.xlsx


In [13]:
diagnostic_file = (
    bronze_dir
    / "2026-10-22T112905699252Z"
    / "1"
    / "diagnostics"
    / "2026-10-01_2026-10-15_capped.xlsx"
)

raw_df = pd.read_excel(diagnostic_file)

print("Shape:", raw_df.shape)
print("\nColumns:")
print(raw_df.columns.tolist())

raw_df.head()

Shape: (50, 10)

Columns:
['Market', 'Commodity', 'Classification', 'Grade', 'Sex', 'Wholesale', 'Retail', 'Supply Volume', 'County', 'Date']


,Market,Commodity,Classification,Grade,Sex,Wholesale,Retail,Supply Volume,County,Date
0,Katito,Dry Maize,White Maize,-,-,50.00/Kg,61.90/Kg,7800.0,Kisumu,2026-10-15
1,Kipkaren,Dry Maize,White Maize,-,-,60.00/Kg,80.00/Kg,5000.0,Kakamega,2026-10-15
2,Maua,Dry Maize,White Maize,-,-,50.00/Kg,60.00/Kg,7200.0,Meru,2026-10-15
3,Eldoret Main,Dry Maize,White Maize,-,-,50.00/Kg,55.00/Kg,270.0,Uasin-Gishu,2026-10-15
4,Kakamega Town,Dry Maize,White Maize,-,-,44.44/Kg,60.00/Kg,20000.0,Kakamega,2026-10-14


In [16]:
kawangware_rows = raw_df[
    (raw_df["Market"] == "Kawangware")
    & (
        pd.to_datetime(raw_df["Date"]).dt.date
        == pd.Timestamp("2026-10-05").date()
    )
]

print("Matching rows:", len(kawangware_rows))

kawangware_rows

Matching rows: 0


,Market,Commodity,Classification,Grade,Sex,Wholesale,Retail,Supply Volume,County,Date
